# Initial Train Dataset Build (Ingestion-Only)
Runs: WHO API probe -> X load -> Y fetch -> Polars preprocessing.

In [ ]:
from pathlib import Path
import subprocess
import polars as pl

In [ ]:
repo = Path.cwd().resolve()
if not (repo / 'ml' / 'scripts').exists():
    repo = repo.parent.parent
steps = [
    ['python', 'ml/scripts/probe_who_news_apis.py', '--out', 'ml/data/raw/who_api_probe.json'],
    ['python', 'ml/scripts/load_x_from_db.py', '--db-path', 'backend/app.db', '--out', 'ml/data/processed/x_candidates.parquet'],
    ['python', 'ml/scripts/fetch_y_who_news.py', '--out-json', 'ml/data/raw/y_who_news_raw.json', '--out-parquet', 'ml/data/processed/y_candidates.parquet', '--top', '100'],
    ['python', 'ml/scripts/preprocess_xy_polars.py', '--x', 'ml/data/processed/x_candidates.parquet', '--y', 'ml/data/processed/y_candidates.parquet', '--out', 'ml/data/processed/ml_ready.parquet'],
]
for cmd in steps:
    print('running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=repo, check=True)


In [ ]:
df = pl.read_parquet('ml/data/processed/ml_ready.parquet')
print('rows', df.height)
print('columns', df.columns)
df.select(['y_source', 'publication_ts', 'risk_rating_norm', 'y_grade3_plus', 'y_event_start_t24h', 'y_event_start_t72h', 'x_value_mean', 'x_row_count']).head(10)